# Machine Translation Transformer


## Setup


In [1]:

# !pip install -q sentencepiece sacrebleu transformers datasets accelerate matplotlib pandas
from sklearn.model_selection import train_test_split
import copy
import math
import os
import random
import re
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

Device: cuda
PyTorch: 2.10.0+cu128


In [2]:
@dataclass
class Config:
    data_path: str = "data.txt"
    max_pairs: Optional[int] = 60_000
    max_length: int = 96
    vocab_size: int = 8_000
    val_fraction: float = 0.05
    test_fraction: float = 0.05

    d_model: int = 256
    num_heads: int = 8
    num_encoder_layers: int = 3
    num_decoder_layers: int = 3
    d_ff: int = 1024
    dropout: float = 0.1

    batch_size: int = 64
    epochs: int = 8
    learning_rate: float = 3e-4
    weight_decay: float = 1e-2
    grad_clip: float = 1.0
    label_smoothing: float = 0.0
    num_workers: int = 2

CFG = Config()

QUICK_MODE = False
if QUICK_MODE:
    CFG.max_pairs = 8_000
    CFG.d_model = 128
    CFG.num_heads = 4
    CFG.num_encoder_layers = 2
    CFG.num_decoder_layers = 2
    CFG.d_ff = 512
    CFG.batch_size = 32
    CFG.epochs = 3

CFG

Config(data_path='data.txt', max_pairs=60000, max_length=96, vocab_size=8000, val_fraction=0.05, test_fraction=0.05, d_model=256, num_heads=8, num_encoder_layers=3, num_decoder_layers=3, d_ff=1024, dropout=0.1, batch_size=64, epochs=8, learning_rate=0.0003, weight_decay=0.01, grad_clip=1.0, label_smoothing=0.0, num_workers=2)

## Data Pipeline


In [3]:
signs = '.,?!:; '
numbers = ''.join([str(i) for i in range(10)])
RUSSIAN_LETTERS_AND_SIGNS = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя' + signs + numbers
ENGLISH_LETTERS_AND_SIGNS = ''.join([chr(i) for i in range(ord('a'), ord('z') + 1)]) + signs + numbers


def normalize_text(text: str) -> str:
    """Minimal normalization that preserves useful information."""
    res_text = " ".join(text.strip().split())
    
    return res_text


def load_parallel_corpus(
    path: str,
    max_pairs: Optional[int] = None,
) -> list[tuple[str, str]]:
    """Return (source_text, target_text) pairs."""
    pairs = []
    seen = set()
    bad_format = 0
    empty_lines = 0
    duplicates = 0
    with open(path, encoding="utf-8") as f:

        for line in f:
            parts = line.rstrip("\n").split('\t')
            
            if len(parts) != 2:
                bad_format += 1
                continue

            src = normalize_text(parts[0])
            tgt = normalize_text(parts[1])

            if not src or not tgt:
                empty_lines += 1
                continue

            pair = (src, tgt)

            if pair not in seen:
                seen.add(pair)
                pairs.append(pair)
            else:
                duplicates += 1
                continue

            if max_pairs is not None and max_pairs <= len(pairs):
                break

    print(
        f"Dropped rows: bad_format={bad_format}, "
        f"empty={empty_lines}, duplicates={duplicates}"
    )
    return pairs


In [4]:
def split_parallel_data(
    pairs: list[tuple[str, str]],
    val_fraction: float,
    test_fraction: float,
    seed: int,
) -> tuple[
    list[tuple[str, str]],
    list[tuple[str, str]],
    list[tuple[str, str]],
]:
    train_val, test = train_test_split(pairs, test_size = test_fraction, random_state=seed, shuffle = True)
    rel_val = val_fraction / (1 - test_fraction)
    train, val = train_test_split(train_val, test_size=rel_val, random_state=seed, shuffle=True)



    return train, val, test

In [5]:
import sentencepiece as spm

def train_sentencepiece(
    train_pairs: list[tuple[str, str]],
    model_prefix: str,
    vocab_size: int,
) -> str:
    """
    Создаёт train-only текстовый файл, обучает shared BPE
    и возвращает путь к .model.
    """
    input_path = model_prefix + "_train.txt"


    with open(input_path, "w", encoding="utf-8") as f:
        for src, tgt in train_pairs:
            f.write(src + '\n')
            f.write(tgt + '\n')



    
    spm.SentencePieceTrainer.train(input = input_path, model_prefix = model_prefix, vocab_size = vocab_size, model_type = 'bpe', character_coverage=1.0, pad_id=0, unk_id=1, bos_id=2, eos_id=3, )


    return model_prefix + ".model"



In [6]:
class ParallelTextDataset(Dataset):
    def __init__(self, pairs: list[tuple[str, str]]):
        self.pairs = pairs

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> tuple[str, str]:
        return self.pairs[index]


def encode_with_special_tokens(
    text: str,
    tokenizer,
    max_length: int,
) -> list[int]:
    """Формат: [BOS] + pieces + [EOS]."""
    ids = tokenizer.encode(text, out_type = int)
    eos_id = tokenizer.eos_id()
    bos_id = tokenizer.bos_id()

    ids = [bos_id] + ids + [eos_id]
    
    if len(ids) > max_length:
        ids = ids[:max_length]
        ids[-1] = eos_id
     
    return ids


def make_collate_fn(tokenizer, max_length: int):
    pad_id = tokenizer.pad_id()

    def collate(batch: list[tuple[str, str]]) -> dict[str, torch.Tensor]:
        src_res = []
        tgt_res = []
        padded_src = []
        padded_tgt = []

        for src, tgt in batch:
            src_res.append(encode_with_special_tokens(src, tokenizer, max_length))
            tgt_res.append(encode_with_special_tokens(tgt, tokenizer, max_length))

        max_tgt = max(len(seq) for seq in tgt_res)
        max_src = max(len(seq) for seq in src_res)

        for seq in src_res:
            padded = seq + [pad_id]*(max_src - len(seq))
            padded_src.append(padded)



        for seq in tgt_res:
            padded = seq + [pad_id]*(max_tgt - len(seq))
            padded_tgt.append(padded)

        src_tsr = torch.tensor(padded_src)
        tgt_tsr = torch.tensor(padded_tgt)

        res_dict = {
            "src": src_tsr,
            "tgt": tgt_tsr
        }

        return res_dict

    return collate

## Masks


In [7]:
def shift_target(tgt: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """
    tgt: [B, T], содержит BOS и EOS.
    Возвращает decoder_input и decoder_target длины T - 1.
    """
    dec_input = tgt[:, :-1]

    dec_tgt = tgt[:, 1:]

    return dec_input, dec_tgt

In [8]:
def make_src_mask(src: torch.Tensor, pad_id: int) -> torch.Tensor:

    mask = src != pad_id
    mask = mask.unsqueeze(1)
    mask = mask.unsqueeze(2)
    
    return mask


def make_causal_mask(length: int, device: torch.device) -> torch.Tensor:
    
    mask = torch.ones((length,length), dtype=torch.bool, device=device)
    mask = torch.tril(mask)
 
    return mask


def make_tgt_mask(tgt_in: torch.Tensor, pad_id: int) -> torch.Tensor:
    mask = tgt_in != pad_id
    size = mask.size(1)
    mask = mask.unsqueeze(1)
    mask = mask.unsqueeze(2)
    cas_mask = make_causal_mask(size, tgt_in.device)
    cas_mask = cas_mask.unsqueeze(0)
    cas_mask = cas_mask.unsqueeze(0)
    
    res_tgt_mask = cas_mask & mask
    return res_tgt_mask

In [9]:
def test_masks():
    pad_id = 0
    src = torch.tensor([[2, 5, 6, 3, 0], [2, 7, 3, 0, 0]])
    tgt = torch.tensor([[2, 9, 8, 3], [2, 4, 0, 0]])

    src_mask = make_src_mask(src, pad_id)
    tgt_mask = make_tgt_mask(tgt, pad_id)

    assert src_mask.shape == (2, 1, 1, 5)
    assert tgt_mask.shape == (2, 1, 4, 4)
    assert src_mask.dtype == torch.bool
    assert tgt_mask.dtype == torch.bool

    assert not tgt_mask[0, 0, 0, 1]
    assert not tgt_mask[0, 0, 1, 2]
    assert tgt_mask[0, 0, 2, 0]
    assert tgt_mask[0, 0, 2, 2]
    assert not tgt_mask[1, 0, :, 2].any()
    assert not tgt_mask[1, 0, :, 3].any()

    print("Mask tests passed.")

test_masks()

Mask tests passed.


## Attention


In [10]:
def scaled_dot_product_attention(
    q: torch.Tensor,
    k: torch.Tensor,
    v: torch.Tensor,
    mask: Optional[torch.Tensor] = None,
    dropout: Optional[nn.Dropout] = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    # сделать емае (q,k,v уже включают в себя вектора эмбеддингов)
    d = q.size(-1) ** 0.5
    scores = q @ k.transpose(2,3)
    scores = scores / d
    
    if mask is not None:
        masked_scores = scores.masked_fill(~mask, float("-inf"))
    else:
        masked_scores = scores

    weights = F.softmax(masked_scores, dim = -1)

    if dropout is not None:
        weights = dropout(weights)

    dot_product = weights @ v
    return dot_product, weights

In [11]:
def test_scaled_dot_product_attention():
    q = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
    k = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])
    v = torch.tensor([[[[10.0, 0.0], [0.0, 20.0]]]])
    mask = torch.tensor([[[[True, False], [True, True]]]])

    out, weights = scaled_dot_product_attention(q, k, v, mask=mask)

    assert out.shape == (1, 1, 2, 2)
    assert weights.shape == (1, 1, 2, 2)
    assert torch.allclose(
        weights[0, 0, 0],
        torch.tensor([1.0, 0.0]),
        atol=1e-6,
    )
    assert torch.isfinite(out).all()
    assert torch.isfinite(weights).all()
    print("Scaled attention tests passed.")

test_scaled_dot_product_attention()

Scaled attention tests passed.


In [12]:
class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_model: int,
        num_heads: int,
        dropout: float = 0.1,
        bias: bool = True,
    ):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self._dropout = nn.Dropout(dropout)
        self.bias = bias 
        self.W_Q = nn.Linear(d_model, d_model, bias=True)
        self.W_K = nn.Linear(d_model, d_model, bias=True)
        self.W_V = nn.Linear(d_model, d_model, bias=True)
        self.W_O = nn.Linear(d_model, d_model, bias = True)
        assert d_model % num_heads == 0
        

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B,L,D] -> [B,H,L,Dh]."""
        B, L, D = x.shape
        d_head = x.size(-1) // self.num_heads
        x = x.view(B, L, self.num_heads, d_head)
        x = x.transpose(1, 2)
        return x

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B,H,L,Dh] -> [B,L,D]."""
        B, H, L, Dh = x.shape
        x = x.transpose(1, 2).contiguous()
        x = x.view(B, L, H * Dh)

        return x

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: Optional[torch.Tensor] = None,
        need_weights: bool = False,
    ):
        Q = self.W_Q(query)
        K = self.W_K(key)
        V = self.W_V(value)

        Q_splitted = self._split_heads(Q)
        K_splitted = self._split_heads(K)
        V_splitted = self._split_heads(V)

        
        attention_output, weights = scaled_dot_product_attention(Q_splitted, K_splitted, V_splitted, mask, self._dropout)
        attention_output = self._merge_heads(attention_output)

        res_out = self.W_O(attention_output)

        if need_weights == True:
            return res_out, weights

        return res_out, None

In [13]:
def test_multi_head_attention():
    torch.manual_seed(0)
    module = MultiHeadAttention(d_model=16, num_heads=4, dropout=0.0)
    x = torch.randn(2, 5, 16, requires_grad=True)
    mask = torch.ones(2, 1, 5, 5, dtype=torch.bool)

    out, weights = module(x, x, x, mask=mask, need_weights=True)

    assert out.shape == (2, 5, 16)
    assert weights.shape == (2, 4, 5, 5)
    assert torch.allclose(
        weights.sum(dim=-1),
        torch.ones(2, 4, 5),
        atol=1e-5,
    )

    out.sum().backward()
    assert x.grad is not None
    assert torch.isfinite(x.grad).all()
    print("Multi-head attention tests passed.")

test_multi_head_attention()

Multi-head attention tests passed.


## Transformer


In [14]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_length: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.max_length = max_length
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_length, d_model)

        position = torch.arange(0, max_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # [1, max_length, d_model]

        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B,L,D]
        L = x.size(1)

        x = x + self.pe[:, :L, :]

        return self.dropout(x)

In [15]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.dropout = nn.Dropout(dropout)
        self.W1 = nn.Linear(d_model, d_ff, bias=True)
        self.W2 = nn.Linear(d_ff, d_model, bias=True)
        

    def forward(self, x: torch.Tensor) -> torch.Tensor:

        x = self.W1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.W2(x)
        return x


class EncoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.dropout = nn.Dropout(dropout)
        self.mhead_att = MultiHeadAttention(self.d_model, self.num_heads, dropout, bias=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.fc = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, src_mask: torch.Tensor) -> torch.Tensor:

        out, _= self.mhead_att(x, x, x, src_mask) 
        x = x + self.dropout(out)
        x = self.norm1(x)
        fc_out = self.fc(x)
        x = x + self.dropout(fc_out)
        x = self.norm2(x)
        
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_ff = d_ff
        self.dropout = nn.Dropout(dropout)
        self.att = MultiHeadAttention(self.d_model, self.num_heads, dropout, bias=True)
        self.cross_att = MultiHeadAttention(self.d_model, self.num_heads, dropout, bias=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.fc = PositionWiseFeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(
        self,
        x: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor,
        memory_mask: torch.Tensor,
        need_cross_attention: bool = False,
    ):
        masked_att_out, _ = self.att(x, x, x, mask=tgt_mask)
        x = x + self.dropout(masked_att_out)
        x = self.norm1(x)
        if need_cross_attention == False:
            cross_att_out, _ = self.cross_att(x, memory, memory, mask=memory_mask)
        else:
            cross_att_out, weights = self.cross_att(x, memory, memory, mask=memory_mask, need_weights=True)
            
        x = x + self.dropout(cross_att_out)
        x = self.norm2(x)
        fc_out = self.fc(x)
        x = x + self.dropout(fc_out)
        x = self.norm3(x)
        if need_cross_attention == False:
            return x, None
        return x, weights

In [16]:
class Seq2SeqTransformer(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        pad_id: int,
        d_model: int,
        num_heads: int,
        num_encoder_layers: int,
        num_decoder_layers: int,
        d_ff: int,
        dropout: float,
        max_length: int,
    ):
        super().__init__()
        self.src_vocab_size = src_vocab_size
        self.tgt_vocab_size = tgt_vocab_size
        self.pad_id = pad_id
        self.d_model = d_model
        self.num_heads = num_heads
        self.num_encoder_layers = num_encoder_layers
        self.num_decoder_layers = num_decoder_layers
        self.d_ff = d_ff
        self.dropout = nn.Dropout(dropout)
        self.max_length = max_length
        self.sqrt_emb = d_model ** 0.5

        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_id) 
        self.encoder_layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_encoder_layers)])
        self.pos_encode = SinusoidalPositionalEncoding(d_model, max_length, dropout)

        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_id)
        self.decoder_layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff, dropout) for _ in range(num_decoder_layers)])
        self.pos_decode = SinusoidalPositionalEncoding(d_model, max_length, dropout)

        self.linear = nn.Linear(d_model, tgt_vocab_size)

    def encode(self, src: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        """Возвращает memory и src_mask."""
        src_mask = make_src_mask(src, pad_id=self.pad_id)
        src_embedded = self.src_embedding(src)
        src_embedded = self.sqrt_emb * src_embedded
        src_embedded = self.pos_encode(src_embedded)
        for layer in self.encoder_layers:
            src_embedded = layer(src_embedded, src_mask)

        memory = src_embedded

        return memory, src_mask

    def decode(
        self,
        tgt_in: torch.Tensor,
        memory: torch.Tensor,
        src_mask: torch.Tensor,
    ) -> torch.Tensor:
        tgt_mask = make_tgt_mask(tgt_in, pad_id=self.pad_id)
        tgt_embedded = self.tgt_embedding(tgt_in)
        tgt_embedded = self.sqrt_emb * tgt_embedded
        tgt_embedded = self.pos_decode(tgt_embedded)
        
        for layer in self.decoder_layers:
            tgt_embedded, _ = layer(tgt_embedded, memory, tgt_mask, src_mask, need_cross_attention = False)

        tgt_out = tgt_embedded
        
        return tgt_out

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor) -> torch.Tensor:
        memory, src_mask = self.encode(src)
        dec_out = self.decode(tgt_in, memory, src_mask)
        dec_out = self.linear(dec_out)

        return dec_out


def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    params = model.parameters()
    if trainable_only:
        params = (p for p in params if p.requires_grad)
    return sum(p.numel() for p in params)

In [17]:

def test_transformer_shapes(model: Seq2SeqTransformer, vocab_size: int):
    model.eval()
    src = torch.tensor(
        [[2, 10, 11, 3, 0], [2, 12, 3, 0, 0]],
        device=DEVICE,
    )
    tgt = torch.tensor(
        [[2, 20, 21, 3], [2, 22, 3, 0]],
        device=DEVICE,
    )
    tgt_in, _ = shift_target(tgt)
    with torch.no_grad():
        logits = model(src, tgt_in)
    assert logits.shape == (2, tgt_in.size(1), vocab_size)
    assert torch.isfinite(logits).all()
    print("Transformer shape tests passed.")

model = Seq2SeqTransformer(
    src_vocab_size=50,
    tgt_vocab_size=50,
    pad_id=0,
    d_model=16,
    num_heads=4,
    num_encoder_layers=2,
    num_decoder_layers=2,
    d_ff=64,
    dropout=0.0,
    max_length=128,
).to(DEVICE)

test_transformer_shapes(model, 50)



Transformer shape tests passed.


In [18]:
def test_no_future_leakage(model: Seq2SeqTransformer):
    model.eval()
    src = torch.tensor([[2, 10, 11, 3]], device=DEVICE)

    tgt_a = torch.tensor([[2, 20, 21, 30, 31]], device=DEVICE)
    tgt_b = torch.tensor([[2, 20, 21, 40, 41]], device=DEVICE)

    with torch.no_grad():
        logits_a = model(src, tgt_a)
        logits_b = model(src, tgt_b)

    assert torch.allclose(
        logits_a[:, :3],
        logits_b[:, :3],
        atol=1e-5,
    ), "Будущие target tokens влияют на прошлые позиции."

    print("No-future-leakage test passed.")

test_no_future_leakage(model)

No-future-leakage test passed.


## Training


In [19]:
def compute_token_loss(
    logits: torch.Tensor,
    targets: torch.Tensor,
    pad_id: int,
    label_smoothing: float = 0.0,
) -> tuple[torch.Tensor, int]:
    logits_flat = logits.reshape(-1, logits.size(-1))
    targets_flat = targets.reshape(-1)
    num_unempty_tokens = (targets_flat != pad_id).sum().item()

    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=pad_id, label_smoothing=label_smoothing, reduction = "mean")
    return loss, num_unempty_tokens


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    pad_id: int,
    grad_clip: float,
    label_smoothing: float = 0.0,
) -> float:
    model.train()
    res_loss = 0.0
    total_tokens = 0

    for batch in loader:
        src = batch["src"].to(DEVICE)
        tgt = batch["tgt"].to(DEVICE)

        tgt_in, tgt_target = shift_target(tgt)

        optimizer.zero_grad(set_to_none=True)
        logits = model(src, tgt_in)
        loss, num_unempty_tokens = compute_token_loss(logits, tgt_target, pad_id, label_smoothing)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        res_loss += loss.item() * num_unempty_tokens
        total_tokens += num_unempty_tokens

    return res_loss / total_tokens


@torch.no_grad()
def evaluate_loss(
    model: nn.Module,
    loader: DataLoader,
    pad_id: int,
    label_smoothing: float = 0.0,
) -> float:
    model.eval()
    res_loss = 0.0
    total_tokens = 0

    for batch in loader:
        src = batch["src"].to(DEVICE)
        tgt = batch["tgt"].to(DEVICE)

        tgt_in, tgt_target = shift_target(tgt)

        logits = model(src, tgt_in)
        loss, num_unempty_tokens = compute_token_loss(logits, tgt_target, pad_id, label_smoothing)

        res_loss += loss.item() * num_unempty_tokens
        total_tokens += num_unempty_tokens

    return res_loss / total_tokens


In [20]:
def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    pad_id: int,
    epochs: int,
    checkpoint_path: str,
    grad_clip: float,
    label_smoothing: float = 0.0,
    scheduler=None,
) -> pd.DataFrame:
    history = []
    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        start_time = time.time()
        print(f"Epoch {epoch}/{epochs} started")

        train_loss = train_one_epoch(model, train_loader, optimizer, pad_id, grad_clip, label_smoothing)
        val_loss = evaluate_loss(model, val_loader, pad_id, label_smoothing)

        if scheduler is not None:
            scheduler.step()

        epoch_time = time.time() - start_time

        lr = optimizer.param_groups[0]["lr"]
        is_best = val_loss < best_val_loss

        if is_best:
            best_val_loss = val_loss
            torch.save(
            {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "train_loss": train_loss,
            },
            checkpoint_path,)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "lr": lr,
            "time": epoch_time,
            "best": is_best,
        })

        best_marker = " | saved best" if is_best else ""
        print(
            f"Epoch {epoch}/{epochs} done | "
            f"train_loss={train_loss:.4f} | "
            f"val_loss={val_loss:.4f} | "
            f"best_val={best_val_loss:.4f} | "
            f"lr={lr:.2e} | "
            f"time={epoch_time:.1f}s | "
            f"left={epochs - epoch}"
            f"{best_marker}"
        )

    return pd.DataFrame(history)


## Inference and Evaluation


In [21]:
@torch.no_grad()
def greedy_decode(
    model: Seq2SeqTransformer,
    src: torch.Tensor,
    bos_id: int,
    eos_id: int,
    pad_id: int,
    max_new_tokens: int,
) -> torch.Tensor:
    B = src.size(0)
    generated = torch.full((B, 1), bos_id, dtype=torch.long, device=src.device)
    is_finished = torch.zeros(B, dtype = torch.bool, device = src.device)
    memory, src_mask = model.encode(src)

    for _ in range(max_new_tokens):
        dec_out = model.decode(generated, memory, src_mask)
        logits = model.linear(dec_out)
        new = logits[:, -1, :].argmax(dim = -1)

        new = torch.where(is_finished, torch.tensor(pad_id, device=src.device), new)

        generated = torch.cat([generated, new.unsqueeze(1)], dim = 1)

        is_finished = is_finished | (new == eos_id)

        if is_finished.all():
            break

        

    return generated


In [22]:
import sacrebleu

def decode_token_ids(ids: list[int], tokenizer) -> str:
    """Remove BOS/PAD and stop at EOS."""
    cleaned_ids = []

    bos_id = tokenizer.bos_id()
    eos_id = tokenizer.eos_id()
    pad_id = tokenizer.pad_id()

    for token in ids:
        if token == bos_id:
            continue
        elif token == eos_id:  
            break
        elif token == pad_id:
            continue

        cleaned_ids.append(token)
    
    decoded = tokenizer.decode(cleaned_ids)
    return decoded


@torch.no_grad()
def generate_translations(
    model: Seq2SeqTransformer,
    loader: DataLoader,
    tokenizer,
    max_new_tokens: int,
) -> tuple[list[str], list[str], list[str]]:
    """Return sources, references, hypotheses."""

    model.eval()
    sources = []
    references = []
    hypotheses = []
    total_batches = len(loader)
    total_examples = len(loader.dataset) if hasattr(loader, "dataset") else None

    for batch_index, batch in enumerate(loader, start=1):
        src = batch["src"].to(DEVICE)
        tgt = batch["tgt"].to(DEVICE)

        generated = greedy_decode(
            model,
            src,
            bos_id=tokenizer.bos_id(),
            eos_id=tokenizer.eos_id(),
            pad_id=tokenizer.pad_id(),
            max_new_tokens=max_new_tokens,
        )

        for src_ids, tgt_ids, gen_ids in zip(src, tgt, generated):
            sources.append(decode_token_ids(src_ids.tolist(), tokenizer))
            references.append(decode_token_ids(tgt_ids.tolist(), tokenizer))
            hypotheses.append(decode_token_ids(gen_ids.tolist(), tokenizer))

        processed = len(sources)
        if total_examples is None:
            print(f"Inference batch {batch_index}/{total_batches} | processed={processed}")
        else:
            left = max(total_examples - processed, 0)
            print(
                f"Inference batch {batch_index}/{total_batches} | "
                f"processed={processed}/{total_examples} | left={left}"
            )

    return sources, references, hypotheses


def compute_mt_metrics(
    references: list[str],
    hypotheses: list[str],
    test_loss: Optional[float] = None
) -> dict[str, float]:
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    
    avg_hypothesis_lengt = (sum(len(hyp.split()) for hyp in hypotheses) / len(hypotheses) if hypotheses else 0.0)

    metrics = {
        "bleu": bleu.score,
        "chrf": chrf.score,
        "avg_hypothesis_lengt": avg_hypothesis_lengt,
    }

    if test_loss is not None:
        metrics["test_loss"] = test_loss

    return metrics

def print_experiment_report(
    name: str,
    metrics: dict[str, float],
    examples: pd.DataFrame,
    trainable_params: Optional[int] = None,
    steps: Optional[int] = None,
    history: Optional[pd.DataFrame] = None,
) -> None:
    print(f"\n===== {name} =====")

    if trainable_params is not None:
        print(f"trainable_params: {trainable_params}")

    if steps is not None:
        print(f"steps: {steps}")

    if history is not None and len(history) > 0:
        if "time" in history.columns:
            if "step" in history.columns:
                total_time = float(history["time"].iloc[-1])
            else:
                total_time = float(history["time"].sum())
            print(f"time_sec: {total_time:.2f}")

        print("history:")
        print(history.to_string(index=False))

    print("metrics:")
    print(metrics)

    print("examples:")
    print(examples.to_string(index=False))



In [23]:
pairs = load_parallel_corpus(CFG.data_path, max_pairs=CFG.max_pairs)

train_pairs, val_pairs, test_pairs = split_parallel_data(pairs, val_fraction=CFG.val_fraction, test_fraction=CFG.test_fraction, seed = SEED,)

#TOKENIZERS

model_path = "spm_bpe.model"
if not Path(model_path).exists():
    model_path = train_sentencepiece(train_pairs, model_prefix="spm_bpe", vocab_size=CFG.vocab_size,)

tokenizer = spm.SentencePieceProcessor()
tokenizer.load(model_path)

pad_id = tokenizer.pad_id()
bos_id = tokenizer.bos_id()
eos_id = tokenizer.eos_id()
vocab_size = tokenizer.get_piece_size()

#LOADERS

collate_fn = make_collate_fn(tokenizer, max_length=CFG.max_length)

train_loader = DataLoader(ParallelTextDataset(train_pairs), batch_size=CFG.batch_size, shuffle = True, collate_fn=collate_fn, num_workers=CFG.num_workers,)

val_loader = DataLoader(ParallelTextDataset(val_pairs), batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=CFG.num_workers,)

test_loader = DataLoader(ParallelTextDataset(test_pairs), batch_size=CFG.batch_size, shuffle=False, collate_fn=collate_fn, num_workers=CFG.num_workers,)


#MODEL

model = Seq2SeqTransformer(
    src_vocab_size=vocab_size,
    tgt_vocab_size=vocab_size,
    pad_id=pad_id,
    d_model = CFG.d_model,
    num_heads=CFG.num_heads,
    num_encoder_layers=CFG.num_encoder_layers,
    num_decoder_layers=CFG.num_decoder_layers,
    d_ff = CFG.d_ff,
    dropout = CFG.dropout,
    max_length=CFG.max_length,

).to(DEVICE)

optimizer = torch.optim.AdamW(model.parameters(), lr = CFG.learning_rate, weight_decay=CFG.weight_decay,)


#TRAIN AND GENERATE

history = fit(
    model = model, 
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    pad_id=pad_id,
    epochs=CFG.epochs,
    checkpoint_path="best_transformer.pt",
    grad_clip=CFG.grad_clip,
    label_smoothing=CFG.label_smoothing,
)

scratch_checkpoint = torch.load("best_transformer.pt", map_location=DEVICE)
model.load_state_dict(scratch_checkpoint["model_state_dict"])

sources, references, hypothesis = generate_translations(
    model = model, 
    loader=test_loader,
    tokenizer=tokenizer,
    max_new_tokens=CFG.max_length,
)

scratch_test_loss = evaluate_loss(model, test_loader, pad_id, CFG.label_smoothing)
scratch_metrics = compute_mt_metrics(references, hypothesis, test_loss=scratch_test_loss)

scratch_examples = pd.DataFrame({
    "source": sources[:10],
    "reference": references[:10],
    "hypothesis": hypothesis[:10],
})
print_experiment_report(
    name="Transformer from scratch",
    metrics=scratch_metrics,
    examples=scratch_examples,
    trainable_params=count_parameters(model, trainable_only=True),
    steps=len(train_loader) * CFG.epochs,
    history=history,
)

scratch_metrics, scratch_examples


Dropped rows: bad_format=0, empty=0, duplicates=1775


Epoch 1/8 started


Epoch 1/8 done | train_loss=4.7929 | val_loss=3.7826 | best_val=3.7826 | lr=3.00e-04 | time=18.0s | left=7 | saved best
Epoch 2/8 started


Epoch 2/8 done | train_loss=3.4775 | val_loss=3.1600 | best_val=3.1600 | lr=3.00e-04 | time=17.3s | left=6 | saved best
Epoch 3/8 started


Epoch 3/8 done | train_loss=3.0032 | val_loss=2.8481 | best_val=2.8481 | lr=3.00e-04 | time=17.2s | left=5 | saved best
Epoch 4/8 started


...

Inference batch 36/38 | processed=2304/2412 | left=108


Inference batch 37/38 | processed=2368/2412 | left=44


Inference batch 38/38 | processed=2412/2412 | left=0



===== Transformer from scratch =====
trainable_params: 11681600
steps: 5432
time_sec: 140.00
history:
 epoch  train_loss  val_loss     lr      time  best
     1    4.792879  3.782565 0.0003 18.010681  True
     2    3.477510  3.159978 0.0003 17.346311  True
     3    3.003228  2.848119 0.0003 17.208438  True
     4    2.713550  2.652063 0.0003 17.400116  True
     5    2.506066  2.527520 0.0003 17.948102  True
     6    2.341765  2.435742 0.0003 17.328012  True
     7    2.207654  2.364153 0.0003 17.526025  True
     8    2.090933  2.301661 0.0003 17.228648  True
metrics:
{'bleu': 21.312284126954545, 'chrf': 44.59157516973328, 'avg_hypothesis_lengt': 12.469320066334992, 'test_loss': 2.241231536843435}
examples:
                                                                                                                         source                                                                                                                            reference                  

({'bleu': 21.312284126954545,
  'chrf': 44.59157516973328,
  'avg_hypothesis_lengt': 12.469320066334992,
  'test_loss': 2.241231536843435},
                                               source  \
 0  This card offers many free benefits and discou...   
 1  The kitchen has a dishwasher and an oven, as w...   
 2  Taksim Square, the heart of Istanbul is an 11-...   
 3  The apartments are scattered around a 1000-m2 ...   
 4  Guests are provided with items for a sweet bre...   
 5  Some rooms have a private bathroom, and the sh...   
 6  St Agnes, Perranporth and Truro are around 20 ...   
 7  The property’s restaurant specialises in regio...   
 8  Les 7 Oliviers lies at a distance of 5 km from...   
 9  The spacious bathrooms have fluffy bathrobes a...   
 
                                            reference  \
 0  По ней предоставляется множество бесплатных ус...   
 1  На вилле Mira Sani есть кухня с посудомоечной ...   
 2  До сердца Стамбула - площади Таксим - можно до...   
 3 

## Pretrained Fine-Tuning


In [24]:
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
)

PRETRAINED_MODEL_NAME = "Helsinki-NLP/opus-mt-en-ru"
HF_BATCH_SIZE = 16
HF_LEARNING_RATE = 3e-5
HF_WEIGHT_DECAY = 1e-2
HF_MAX_STEPS = 500
HF_MAX_SOURCE_LENGTH = CFG.max_length
HF_MAX_TARGET_LENGTH = CFG.max_length

hf_tokenizer = AutoTokenizer.from_pretrained(PRETRAINED_MODEL_NAME)


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/803k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/home/jovyan/.mlspace/envs/sentsov-sft/lib/python3.11/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [25]:
def freeze_pretrained_encoder(model: nn.Module) -> nn.Module:
    for parameter in model.model.encoder.parameters():
        parameter.requires_grad = False
    
    return model


def unfreeze_all(model: nn.Module) -> nn.Module:
    for parameter in model.parameters():
        parameter.requires_grad = True

    return model


def count_trainable_params(model: nn.Module) -> int:
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


class HFParallelTextDataset(Dataset):
    def __init__(self, pairs: list[tuple[str, str]]):
        self.pairs = pairs

    def __len__(self) -> int:
        return len(self.pairs)

    def __getitem__(self, index: int) -> dict[str, str]:
        src, tgt = self.pairs[index]
        return {"src": src, "tgt": tgt}


def make_hf_collate_fn(tokenizer, max_source_length: int, max_target_length: int):
    def collate(batch: list[dict[str, str]]) -> dict[str, torch.Tensor]:
        sources = [item["src"] for item in batch]
        targets = [item["tgt"] for item in batch]

        model_inputs = tokenizer(
            sources,
            max_length=max_source_length,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )

        labels = tokenizer(
            text_target=targets,
            max_length=max_target_length,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )["input_ids"]

        labels[labels == tokenizer.pad_token_id] = -100
        model_inputs["labels"] = labels
        return model_inputs

    return collate


def make_hf_loaders(
    train_pairs: list[tuple[str, str]],
    val_pairs: list[tuple[str, str]],
    test_pairs: list[tuple[str, str]],
    tokenizer,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    collate_fn = make_hf_collate_fn(
        tokenizer,
        max_source_length=HF_MAX_SOURCE_LENGTH,
        max_target_length=HF_MAX_TARGET_LENGTH,
    )

    train_loader = DataLoader(
        HFParallelTextDataset(train_pairs),
        batch_size=HF_BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=0,
    )
    val_loader = DataLoader(
        HFParallelTextDataset(val_pairs),
        batch_size=HF_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )
    test_loader = DataLoader(
        HFParallelTextDataset(test_pairs),
        batch_size=HF_BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0,
    )

    return train_loader, val_loader, test_loader


@torch.no_grad()
def evaluate_hf_loss(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    total_loss = 0.0
    total_examples = 0

    for batch in loader:
        batch = {key: value.to(DEVICE) for key, value in batch.items()}
        outputs = model(**batch)
        batch_size = batch["input_ids"].size(0)
        total_loss += outputs.loss.item() * batch_size
        total_examples += batch_size

    return total_loss / total_examples


def fine_tune_hf_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    max_steps: int,
    learning_rate: float,
    weight_decay: float,
    checkpoint_path: str,
) -> pd.DataFrame:
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(
        (parameter for parameter in model.parameters() if parameter.requires_grad),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    history = []
    best_val_loss = float("inf")
    step = 0
    start_time = time.time()

    while step < max_steps:
        model.train()
        for batch in train_loader:
            batch = {key: value.to(DEVICE) for key, value in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
            optimizer.step()

            step += 1

            if step == 1 or step % 50 == 0 or step == max_steps:
                val_loss = evaluate_hf_loss(model, val_loader)
                is_best = val_loss < best_val_loss

                if is_best:
                    best_val_loss = val_loss
                    torch.save(
                        {
                            "step": step,
                            "model_state_dict": model.state_dict(),
                            "optimizer_state_dict": optimizer.state_dict(),
                            "val_loss": val_loss,
                            "train_loss": loss.item(),
                        },
                        checkpoint_path,
                    )

                elapsed = time.time() - start_time
                history.append(
                    {
                        "step": step,
                        "train_loss": loss.item(),
                        "val_loss": val_loss,
                        "lr": optimizer.param_groups[0]["lr"],
                        "time": elapsed,
                        "best": is_best,
                    }
                )

                best_marker = " | saved best" if is_best else ""
                print(
                    f"HF step {step}/{max_steps} | "
                    f"train_loss={loss.item():.4f} | "
                    f"val_loss={val_loss:.4f} | "
                    f"best_val={best_val_loss:.4f} | "
                    f"left={max_steps - step}"
                    f"{best_marker}"
                )

            if step >= max_steps:
                break

    return pd.DataFrame(history)


@torch.no_grad()
def generate_hf_translations(
    model: nn.Module,
    pairs: list[tuple[str, str]],
    tokenizer,
    batch_size: int,
    max_new_tokens: int,
) -> tuple[list[str], list[str], list[str]]:
    model.eval()
    sources = []
    references = []
    hypotheses = []

    for start in range(0, len(pairs), batch_size):
        batch_pairs = pairs[start:start + batch_size]
        batch_sources = [src for src, _ in batch_pairs]
        batch_references = [tgt for _, tgt in batch_pairs]

        encoded = tokenizer(
            batch_sources,
            max_length=HF_MAX_SOURCE_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt",
        )
        encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

        generated_ids = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            num_beams=1,
        )
        batch_hypotheses = tokenizer.batch_decode(
            generated_ids,
            skip_special_tokens=True,
        )

        sources.extend(batch_sources)
        references.extend(batch_references)
        hypotheses.extend(batch_hypotheses)

        processed = min(start + batch_size, len(pairs))
        print(
            f"HF inference processed={processed}/{len(pairs)} | "
            f"left={len(pairs) - processed}"
        )

    return sources, references, hypotheses


In [26]:
hf_train_loader, hf_val_loader, hf_test_loader = make_hf_loaders(
    train_pairs,
    val_pairs,
    test_pairs,
    hf_tokenizer,
)

hf_frozen_model = AutoModelForSeq2SeqLM.from_pretrained(PRETRAINED_MODEL_NAME)
hf_frozen_model = freeze_pretrained_encoder(hf_frozen_model)
print("Frozen encoder trainable params:", count_trainable_params(hf_frozen_model))

hf_frozen_history = fine_tune_hf_model(
    model=hf_frozen_model,
    train_loader=hf_train_loader,
    val_loader=hf_val_loader,
    max_steps=HF_MAX_STEPS,
    learning_rate=HF_LEARNING_RATE,
    weight_decay=HF_WEIGHT_DECAY,
    checkpoint_path="hf_frozen_encoder_best.pt",
)

hf_frozen_checkpoint = torch.load("hf_frozen_encoder_best.pt", map_location=DEVICE)
hf_frozen_model.load_state_dict(hf_frozen_checkpoint["model_state_dict"])

hf_frozen_sources, hf_frozen_references, hf_frozen_hypotheses = generate_hf_translations(
    model=hf_frozen_model,
    pairs=test_pairs,
    tokenizer=hf_tokenizer,
    batch_size=HF_BATCH_SIZE,
    max_new_tokens=HF_MAX_TARGET_LENGTH,
)

hf_frozen_metrics = compute_mt_metrics(
    references=hf_frozen_references,
    hypotheses=hf_frozen_hypotheses,
    test_loss=evaluate_hf_loss(hf_frozen_model, hf_test_loader),
)

hf_frozen_examples = pd.DataFrame({
    "source": hf_frozen_sources[:10],
    "reference": hf_frozen_references[:10],
    "hypothesis": hf_frozen_hypotheses[:10],
})

print_experiment_report(
    name="Pretrained frozen encoder",
    metrics=hf_frozen_metrics,
    examples=hf_frozen_examples,
    trainable_params=count_trainable_params(hf_frozen_model),
    steps=HF_MAX_STEPS,
    history=hf_frozen_history,
)

hf_frozen_metrics, hf_frozen_examples


pytorch_model.bin:   0%|          | 0.00/307M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Frozen encoder trainable params: 25486336


model.safetensors:   0%|          | 0.00/307M [00:00<?, ?B/s]

HF step 1/500 | train_loss=2.1909 | val_loss=1.9612 | best_val=1.9612 | left=499 | saved best


...

HF inference processed=2368/2412 | left=44


HF inference processed=2384/2412 | left=28


HF inference processed=2400/2412 | left=12
HF inference processed=2412/2412 | left=0



===== Pretrained frozen encoder =====
trainable_params: 25486336
steps: 500
time_sec: 65.88
history:
 step  train_loss  val_loss      lr      time  best
    1    2.190935  1.961171 0.00003  3.436704  True
   50    1.147852  1.419572 0.00003  8.159585  True
  100    1.160248  1.286356 0.00003 12.892173  True
  150    1.262846  1.215074 0.00003 17.602909  True
  200    1.194239  1.161185 0.00003 22.339309  True
  250    1.109136  1.121114 0.00003 27.067169  True
  300    0.880192  1.087983 0.00003 31.740215  True
  350    0.957364  1.065346 0.00003 36.457065  True
  400    1.074252  1.047890 0.00003 41.135963  True
  450    0.917975  1.025841 0.00003 61.069602  True
  500    1.359665  1.012673 0.00003 65.884103  True
metrics:
{'bleu': 25.33989536502583, 'chrf': 50.781708329147854, 'avg_hypothesis_lengt': 13.068822553897181, 'test_loss': 1.0007970630628356}
examples:
                                                                                                                         s

({'bleu': 25.33989536502583,
  'chrf': 50.781708329147854,
  'avg_hypothesis_lengt': 13.068822553897181,
  'test_loss': 1.0007970630628356},
                                               source  \
 0  This card offers many free benefits and discou...   
 1  The kitchen has a dishwasher and an oven, as w...   
 2  Taksim Square, the heart of Istanbul is an 11-...   
 3  The apartments are scattered around a 1000-m2 ...   
 4  Guests are provided with items for a sweet bre...   
 5  Some rooms have a private bathroom, and the sh...   
 6  St Agnes, Perranporth and Truro are around 20 ...   
 7  The property’s restaurant specialises in regio...   
 8  Les 7 Oliviers lies at a distance of 5 km from...   
 9  The spacious bathrooms have fluffy bathrobes a...   
 
                                            reference  \
 0  По ней предоставляется множество бесплатных ус...   
 1  На вилле Mira Sani есть кухня с посудомоечной ...   
 2  До сердца Стамбула - площади Таксим - можно до...   
 3

In [27]:
hf_full_model = AutoModelForSeq2SeqLM.from_pretrained(PRETRAINED_MODEL_NAME)
hf_full_model = unfreeze_all(hf_full_model)
print("Full fine-tuning trainable params:", count_trainable_params(hf_full_model))

hf_full_history = fine_tune_hf_model(
    model=hf_full_model,
    train_loader=hf_train_loader,
    val_loader=hf_val_loader,
    max_steps=HF_MAX_STEPS,
    learning_rate=HF_LEARNING_RATE,
    weight_decay=HF_WEIGHT_DECAY,
    checkpoint_path="hf_full_finetune_best.pt",
)

hf_full_checkpoint = torch.load("hf_full_finetune_best.pt", map_location=DEVICE)
hf_full_model.load_state_dict(hf_full_checkpoint["model_state_dict"])

hf_full_sources, hf_full_references, hf_full_hypotheses = generate_hf_translations(
    model=hf_full_model,
    pairs=test_pairs,
    tokenizer=hf_tokenizer,
    batch_size=HF_BATCH_SIZE,
    max_new_tokens=HF_MAX_TARGET_LENGTH,
)

hf_full_metrics = compute_mt_metrics(
    references=hf_full_references,
    hypotheses=hf_full_hypotheses,
    test_loss=evaluate_hf_loss(hf_full_model, hf_test_loader),
)

hf_full_examples = pd.DataFrame({
    "source": hf_full_sources[:10],
    "reference": hf_full_references[:10],
    "hypothesis": hf_full_hypotheses[:10],
})

print_experiment_report(
    name="Pretrained full fine-tuning",
    metrics=hf_full_metrics,
    examples=hf_full_examples,
    trainable_params=count_trainable_params(hf_full_model),
    steps=HF_MAX_STEPS,
    history=hf_full_history,
)

hf_full_metrics, hf_full_examples


Full fine-tuning trainable params: 76672000


HF step 1/500 | train_loss=1.8894 | val_loss=1.9600 | best_val=1.9600 | left=499 | saved best


HF step 50/500 | train_loss=1.0125 | val_loss=1.2617 | best_val=1.2617 | left=450 | saved best


HF step 100/500 | train_loss=1.1820 | val_loss=1.1235 | best_val=1.1235 | left=400 | saved best


HF step 150/500 | train_loss=0.9605 | val_loss=1.0504 | best_val=1.0504 | left=350 | saved best


...

HF inference processed=2384/2412 | left=28


HF inference processed=2400/2412 | left=12


HF inference processed=2412/2412 | left=0



===== Pretrained full fine-tuning =====
trainable_params: 76672000
steps: 500
time_sec: 78.69
history:
 step  train_loss  val_loss      lr      time  best
    1    1.889373  1.960004 0.00003  4.088659  True
   50    1.012514  1.261749 0.00003  9.910690  True
  100    1.182041  1.123511 0.00003 15.830816  True
  150    0.960492  1.050406 0.00003 21.789359  True
  200    1.053560  1.000621 0.00003 27.781446  True
  250    0.901929  0.964062 0.00003 33.673674  True
  300    0.848012  0.943857 0.00003 39.550442  True
  350    0.827946  0.919855 0.00003 45.451496  True
  400    1.088445  0.903793 0.00003 66.604834  True
  450    0.802136  0.892064 0.00003 72.566039  True
  500    0.622708  0.873963 0.00003 78.687663  True
metrics:
{'bleu': 30.53960323685233, 'chrf': 55.72512168670464, 'avg_hypothesis_lengt': 13.230514096185738, 'test_loss': 0.861249872701085}
examples:
                                                                                                                         s

({'bleu': 30.53960323685233,
  'chrf': 55.72512168670464,
  'avg_hypothesis_lengt': 13.230514096185738,
  'test_loss': 0.861249872701085},
                                               source  \
 0  This card offers many free benefits and discou...   
 1  The kitchen has a dishwasher and an oven, as w...   
 2  Taksim Square, the heart of Istanbul is an 11-...   
 3  The apartments are scattered around a 1000-m2 ...   
 4  Guests are provided with items for a sweet bre...   
 5  Some rooms have a private bathroom, and the sh...   
 6  St Agnes, Perranporth and Truro are around 20 ...   
 7  The property’s restaurant specialises in regio...   
 8  Les 7 Oliviers lies at a distance of 5 km from...   
 9  The spacious bathrooms have fluffy bathrobes a...   
 
                                            reference  \
 0  По ней предоставляется множество бесплатных ус...   
 1  На вилле Mira Sani есть кухня с посудомоечной ...   
 2  До сердца Стамбула - площади Таксим - можно до...   
 3  